# Grok-multimodal FS00-FS02 (hardened acceptance)


In [ ]:
import os, json, math, random, time, re, string
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
os.environ.pop('CUDA_VISIBLE_DEVICES', None)
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT=Path('/kaggle/working'); FIG=OUT/'figures'; RES=OUT/'results'
FIG.mkdir(parents=True, exist_ok=True); RES.mkdir(parents=True, exist_ok=True)
device=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device', device, 'gpus', torch.cuda.device_count() if torch.cuda.is_available() else 0)
PROGRESS={}; GATES={}

def gate(name, ok, detail=''):
    GATES[name]=bool(ok)
    print(('PASS' if ok else 'FAIL'), name, detail)
    if not ok:
        raise AssertionError(f'ACCEPTANCE FAILED: {name} {detail}')

def make_shape_image(kind, size=64):
    img=np.ones((size,size,3), np.float32)*0.95
    yy,xx=np.mgrid[0:size,0:size]
    cy,cx=size//2, size//2
    if kind=='red_circle':
        m=(yy-cy)**2+(xx-cx)**2 <= (size*0.28)**2
        img[m]=(0.9,0.15,0.12)
    elif kind=='blue_square':
        m=(np.abs(yy-cy)<size*0.25)&(np.abs(xx-cx)<size*0.25)
        img[m]=(0.15,0.25,0.85)
    elif kind=='green_triangle':
        # filled upward triangle with lower fill ratio
        top=cy-int(size*0.28); bot=cy+int(size*0.30)
        for y in range(top, bot):
            half=int((y-top)/(bot-top+1e-6)*size*0.30)
            x0=cx-half; x1=cx+half
            img[y, max(0,x0):min(size,x1+1)]=(0.15,0.75,0.25)
    elif kind=='yellow_circle':
        m=(yy-cy)**2+(xx-cx)**2 <= (size*0.28)**2
        img[m]=(0.95,0.85,0.1)
    else:
        raise ValueError(kind)
    return img


## FS00


In [ ]:
# FS00 alignment table
pairs=[
    ('red_circle','a red circle'),
    ('blue_square','a blue square'),
    ('green_triangle','a green triangle'),
    ('red_circle','a blue square'),
]
rows=[]
for kind,txt in pairs:
    img=make_shape_image(kind)
    color_word={'red':0,'blue':1,'green':2}
    shape_word={'circle':0,'square':1,'triangle':2}
    img_id={'red_circle':(0,0),'blue_square':(1,1),'green_triangle':(2,2)}[kind]
    tw=txt.split()
    t_color=next((color_word[w] for w in tw if w in color_word), -1)
    t_shape=next((shape_word[w] for w in tw if w in shape_word), -1)
    align=int(t_color==img_id[0])+int(t_shape==img_id[1])
    rows.append({'image':kind,'text':txt,'hand_align_0_2':align,'match':align==2})
print(rows)
matched=[r for r in rows if r['match']]
mismatched=[r for r in rows if not r['match']]
gate('FS00_matched_score2', all(r['hand_align_0_2']==2 for r in matched))
gate('FS00_mismatch_lt2', all(r['hand_align_0_2']<2 for r in mismatched))
fig,axes=plt.subplots(1,3,figsize=(9,3))
for ax,k in zip(axes,['red_circle','blue_square','green_triangle']):
    ax.imshow(make_shape_image(k)); ax.set_title(k); ax.axis('off')
fig.tight_layout(); fig.savefig(FIG/'fs00_modalities.png', dpi=120); plt.close()
(RES/'fs00.json').write_text(json.dumps({'stage':'FS00','rows':rows,'figure':'figures/fs00_modalities.png'},indent=2))
PROGRESS['FS00']='ok'


## FS01


In [ ]:
# FS01 robust bag-of-colors + geometry (use mask stats carefully)

def bag_of_colors_caption(img):
    # color via nearest prototype on non-background pixels
    bg = img.mean(axis=2) > 0.92
    ink = ~bg
    if ink.sum() < 10:
        return 'a blank scene', [0,0,0], 'blank'
    pix = img[ink]
    mean = pix.mean(axis=0)
    centers = {
        'red': np.array([0.9,0.15,0.12], np.float32),
        'blue': np.array([0.15,0.25,0.85], np.float32),
        'green': np.array([0.15,0.75,0.25], np.float32),
    }
    name = min(centers, key=lambda n: float(np.linalg.norm(mean-centers[n])))
    ys, xs = np.where(ink)
    bw = xs.max()-xs.min()+1; bh = ys.max()-ys.min()+1
    fill = float(ink.sum()) / float(bw*bh)
    # circularity: area vs perimeter box; also radial variance
    cy, cx = ys.mean(), xs.mean()
    rr = np.sqrt((ys-cy)**2 + (xs-cx)**2)
    rstd = float(rr.std() / (rr.mean()+1e-6))
    # square: high fill + low radial variance relative? actually square corners raise rstd slightly
    # circle: fill ~ pi/4=0.785, low rstd
    # triangle: fill ~0.5, higher asymmetry
    if fill >= 0.90:
        shape='square'
    elif fill >= 0.70 and rstd < 0.28:
        shape='circle'
    elif fill < 0.70:
        shape='triangle'
    else:
        # fallback by fill
        shape='circle' if fill < 0.88 else 'square'
    return f'a {name} {shape}', mean.tolist(), shape

inputs=['red_circle','blue_square','green_triangle']
rows=[]
fig,axes=plt.subplots(1,3,figsize=(9,3.2))
for ax,k in zip(axes,inputs):
    img=make_shape_image(k)
    cap,mean,shape=bag_of_colors_caption(img)
    gt='a '+k.replace('_',' ')
    ok = (cap==gt)
    rows.append({'input':k,'gt':gt,'pred':cap,'exact_match':ok,'mean_rgb':[round(x,3) for x in mean],'shape':shape})
    ax.imshow(img); ax.set_title(f'{cap}\n({gt})',fontsize=9); ax.axis('off')
fig.tight_layout(); fig.savefig(FIG/'fs01_captions.png',dpi=120); plt.close()
acc=sum(r['exact_match'] for r in rows)/len(rows)
print(rows, 'acc', acc)
gate('FS01_exact_match', acc>=0.999, f'acc={acc} rows={rows}')
(RES/'fs01.json').write_text(json.dumps({'stage':'FS01','method':'bag-of-colors+geometry','exact_match_acc':acc,'rows':rows,'vs_prev':'FS00 scores; FS01 generates'},indent=2))
PROGRESS['FS01']='ok'


## FS02


In [ ]:
CLASSES=['red_circle','blue_square','green_triangle']
c2i={c:i for i,c in enumerate(CLASSES)}

def synth_dataset(n_per=300, size=32, noise=0.05):
    xs,ys=[],[]
    for c in CLASSES:
        for _ in range(n_per):
            img=make_shape_image(c,64)[::2,::2][:size,:size]
            img=np.clip(img+np.random.randn(*img.shape).astype(np.float32)*noise,0,1)
            xs.append(img.transpose(2,0,1)); ys.append(c2i[c])
    return torch.tensor(np.stack(xs),dtype=torch.float32), torch.tensor(ys)

class TinyCNN(nn.Module):
    def __init__(self,n=3):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(3,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),nn.Linear(64,n))
    def forward(self,x): return self.net(x)

x,y=synth_dataset(); perm=torch.randperm(len(x)); x,y=x[perm],y[perm]
ntr=int(0.8*len(x))
tr=DataLoader(TensorDataset(x[:ntr],y[:ntr]),batch_size=64,shuffle=True)
va=DataLoader(TensorDataset(x[ntr:],y[ntr:]),batch_size=128)
model=TinyCNN().to(device); opt=torch.optim.Adam(model.parameters(),lr=1e-3)
hist=[]
for epoch in range(1,10):
    model.train(); tl=n=0
    for xb,yb in tr:
        xb,yb=xb.to(device),yb.to(device)
        opt.zero_grad(set_to_none=True)
        loss=F.cross_entropy(model(xb),yb); loss.backward(); opt.step()
        tl+=loss.item()*xb.size(0); n+=xb.size(0)
    model.eval(); c=t=0
    with torch.no_grad():
        for xb,yb in va:
            xb,yb=xb.to(device),yb.to(device)
            c+=(model(xb).argmax(1)==yb).sum().item(); t+=yb.size(0)
    hist.append({'epoch':epoch,'train_loss':round(tl/n,4),'val_acc':round(c/t,4)}); print(hist[-1])

# clean eval
ok=0
for k in CLASSES:
    img=make_shape_image(k,64)[::2,::2][:32,:32]
    t=torch.tensor(img.transpose(2,0,1)[None],dtype=torch.float32,device=device)
    with torch.no_grad():
        pred=CLASSES[model(t).argmax(1).item()]
    ok += int(pred==k)
    print('clean',k,'->',pred)
clean_acc=ok/len(CLASSES)
gate('FS02_val_acc', hist[-1]['val_acc']>=0.95, hist[-1])
gate('FS02_clean_acc', clean_acc>=0.999, f'clean_acc={clean_acc}')
fig,axes=plt.subplots(1,3,figsize=(9,3))
for ax,k in zip(axes,CLASSES):
    ax.imshow(make_shape_image(k)); ax.axis('off'); ax.set_title(k)
fig.tight_layout(); fig.savefig(FIG/'fs02_cnn_words.png',dpi=120); plt.close()
(RES/'fs02.json').write_text(json.dumps({'stage':'FS02','method':'TinyCNN','val_acc':hist[-1]['val_acc'],'clean_acc':clean_acc,'history':hist,'vs_prev':'FS01 rules; FS02 learns'},indent=2))
PROGRESS['FS02']='ok'


In [ ]:
(RES/'summary_fs00_fs02.json').write_text(json.dumps({'progress':PROGRESS,'gates':GATES,'device':str(device)},indent=2))
(OUT/'SUCCESS').write_text('ok\n')
(OUT/'ACCEPTANCE.json').write_text(json.dumps({'ok':all(GATES.values()),'gates':GATES},indent=2))
print('FS00-02 ALL PASS', GATES)
